<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/03_deep_learning/architectures/cnn/experiment_cnn_image_classification_custom_dataset_binary_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Required Libraries
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

# --- Start of Proposed Fix ---
# Set Path to Your Dataset
# Place your dataset in this folder structure: dataset/train/[class folders] and dataset/val/[class folders]
# The original paths pointed to /content/sample_data which did not contain images in the expected structure.
# We will create a dummy dataset structure for demonstration purposes.

base_dataset_dir = '/content/image_dataset' # Using a new directory name to avoid conflict with sample_data
train_dir = os.path.join(base_dataset_dir, 'train')
val_dir = os.path.join(base_dataset_dir, 'val')

# Define classes for the dummy dataset
classes = ['class_a', 'class_b'] # Using generic names

# Create directories if they don't exist
for dir_path in [train_dir, val_dir]:
    for cls in classes:
        os.makedirs(os.path.join(dir_path, cls), exist_ok=True)

# Create dummy images
# For training set: 5 images per class
for cls in classes:
    for i in range(5):
        dummy_image_path = os.path.join(train_dir, cls, f'{cls}_{i}.png')
        # Create a simple blank image
        dummy_image = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
        tf.keras.utils.save_img(dummy_image_path, dummy_image)

# For validation set: 2 images per class
for cls in classes:
    for i in range(2):
        dummy_image_path = os.path.join(val_dir, cls, f'{cls}_{i}.png')
        # Create a simple blank image
        dummy_image = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
        tf.keras.utils.save_img(dummy_image_path, dummy_image)

print(f"Dummy dataset structure created at: {base_dataset_dir}")
# --- End of Proposed Fix ---

#  Data Preprocessing
img_size = (128, 128)  # Resize all images to 128x128
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary' if len(os.listdir(train_dir)) == 2 else 'categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary' if len(os.listdir(val_dir)) == 2 else 'categorical'
)

# Build CNN Model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_size[0], img_size[1], 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1 if train_data.num_classes == 2 else train_data.num_classes,
                 activation='sigmoid' if train_data.num_classes == 2 else 'softmax')
])

# Compile the Model
model.compile(
    loss='binary_crossentropy' if train_data.num_classes == 2 else 'categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Train the Model
history = model.fit(
    train_data,
    epochs=10,
    validation_data=val_data
)

# Evaluate the Model
val_loss, val_accuracy = model.evaluate(val_data)
print(f"Validation Accuracy: {val_accuracy:.2f}")

Dummy dataset structure created at: /content/image_dataset
Found 10 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6912 - val_accuracy: 0.5000 - val_loss: 0.8731
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - accuracy: 0.5000 - loss: 0.7394 - val_accuracy: 0.5000 - val_loss: 2.4054
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step - accuracy: 0.5000 - loss: 2.2914 - val_accuracy: 0.5000 - val_loss: 0.7070
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step - accuracy: 0.5000 - loss: 0.5950 - val_accuracy: 0.5000 - val_loss: 1.1803
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step - accuracy: 0.5000 - loss: 1.0736 - val_accuracy: 0.5000 - val_loss: 0.8400
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step - accuracy: 0.5000 - loss: 0.7597 - val_accuracy: 0.5000 - val_loss: 0.6951
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step - accuracy: 0.6000 - loss: 0.6369 - val_accuracy: 0.5000 - val_loss: 0.7144
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step - accuracy: 0.5000 - loss: 0.6713 - val_accuracy: 0.5000 - val_loss: 0.